In [65]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
import joblib

In [66]:
data = pd.read_csv('platform_data.csv', sep=';', encoding='utf-8')
X = data.drop(['name', 'trash'], axis=1)
y = data['trash']

print(f"Размер данных: {X.shape}")
print(f"Баланс классов: 0={sum(y==0)}, 1={sum(y==1)}")
print(f"Доля класса 1: {sum(y==1)/len(y)*100:.2f}%")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=69, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
joblib.dump(scaler, 'scaler.pkl')

Размер данных: (1650, 6)
Баланс классов: 0=592, 1=1058
Доля класса 1: 64.12%


['scaler.pkl']

In [67]:
class NN(nn.Module):
    def __init__(self, input_size):
        super(NN, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x).squeeze()

In [68]:
class Dataset_tensor(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.FloatTensor(labels)
    def __len__(self):
        return len(self.features)
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

In [73]:
def train(model, train_loader, val_loader, epochs=200):
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    best_f1 = 0

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        model.eval()
        val_preds = []
        val_true = []
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                outputs = model(batch_x)
                preds = (outputs > 0.5).float()
                val_preds.extend(preds.numpy())
                val_true.extend(batch_y.numpy())
        val_acc = accuracy_score(val_true, val_preds)
        val_f1 = f1_score(val_true, val_preds)

        if val_f1 > best_f1:
            best_f1 = val_f1
            patience_counter = 0
            torch.save(model.state_dict(), 'best_nn_model.pth')
        else:
            patience_counter += 1

        if (epoch + 1) % 20 == 0:
            print(f"  Эпоха {epoch+1}: Loss={train_loss/len(train_loader):.4f}, "
                  f"Val Acc={val_acc:.4f}, Val F1={val_f1:.4f}")

    return model

In [70]:
def train_catboost(X_train, y_train, X_test, y_test):
    print("  Обучение CatBoost...")
    model = CatBoostClassifier(
        iterations=1000,
        learning_rate=0.05,
        depth=6,
        verbose=0,
        random_state=69
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    return model, acc, f1

In [71]:
def train_xgboost(X_train, y_train, X_test, y_test):
    model = XGBClassifier(
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        random_state=42,
        use_label_encoder=False,
        eval_metric='logloss'
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    return model, acc, f1

In [74]:
def main():
    train_dataset = Dataset_tensor(X_train_scaled, y_train.values)
    test_dataset = Dataset_tensor(X_test_scaled, y_test.values)

    train_size = int(0.8 * len(train_dataset))
    val_size = len(train_dataset) - train_size
    train_subset, val_subset = torch.utils.data.random_split(
        train_dataset, [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )

    train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=32, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    input_size = X_train_scaled.shape[1]
    nn_model = NN(input_size)

    nn_model = train(nn_model, train_loader, val_loader, epochs=100)

    nn_model.load_state_dict(torch.load('best_nn_model.pth'))
    nn_model.eval()

    print("  Тестирование нейросети...")
    y_pred_nn = []
    y_true_nn = []
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            outputs = nn_model(batch_x)
            preds = (outputs > 0.5).float()
            y_pred_nn.extend(preds.numpy())
            y_true_nn.extend(batch_y.numpy())

    nn_acc = accuracy_score(y_true_nn, y_pred_nn)
    nn_f1 = f1_score(y_true_nn, y_pred_nn)

    print(f"  NN: Accuracy={nn_acc:.4f}, F1-Score={nn_f1:.4f}")

    cb_model, cb_acc, cb_f1 = train_catboost(
        X_train_scaled, y_train,
        X_test_scaled, y_test
    )
    print(f"  CatBoost: Accuracy={cb_acc:.4f}, F1-Score={cb_f1:.4f}")

    xgb_model, xgb_acc, xgb_f1 = train_xgboost(
        X_train_scaled, y_train,
        X_test_scaled, y_test
    )
    print(f"  XGBoost: Accuracy={xgb_acc:.4f}, F1-Score={xgb_f1:.4f}")

    print("СРАВНЕНИЕ РЕЗУЛЬТАТОВ")

    print(f"{'Модель':<15} {'Accuracy':<12} {'F1-Score':<12}")
    print("-"*50)
    print(f"{'NN':<15} {nn_acc:<12.4f} {nn_f1:<12.4f}")
    print(f"{'CatBoost':<15} {cb_acc:<12.4f} {cb_f1:<12.4f}")
    print(f"{'XGBoost':<15} {xgb_acc:<12.4f} {xgb_f1:<12.4f}")
    print("="*50)

    print("\n6. Сохранение моделей...")

    results = {
        'NN': {'model': nn_model, 'acc': nn_acc, 'f1': nn_f1},
        'CatBoost': {'model': cb_model, 'acc': cb_acc, 'f1': cb_f1},
        'XGBoost': {'model': xgb_model, 'acc': xgb_acc, 'f1': xgb_f1}
    }

    best_model_name = max(results.items(), key=lambda x: x[1]['f1'])[0]
    best_model = results[best_model_name]['model']

    if best_model_name == 'NN':
        # Сохраняем состояние нейросети
        torch.save({
            'model_state_dict': best_model.state_dict(),
            'input_size': input_size
        }, 'best_model.pth')
    else:
        # Сохраняем модель CatBoost или XGBoost
        joblib.dump(best_model, 'best_model.pkl')

    print(f"  Лучшая модель: {best_model_name} (F1={results[best_model_name]['f1']:.4f})")

    # Сравнение с предыдущими результатами (если есть)
    if best_model_name != 'NN':
        diff_f1 = max(cb_f1, xgb_f1) - nn_f1
        diff_acc = max(cb_acc, xgb_acc) - nn_acc

        print(f"  • Деревья лучше нейросети на:")
        print(f"    - F1-Score: +{diff_f1:.4f}")
        print(f"    - Accuracy: +{diff_acc:.4f}")

    if best_model_name == 'NN':
        print("  • NN показала лучший результат!")
    elif best_model_name == 'CatBoost':
        print("  • CatBoost показал лучший результат")
    else:
        print("  • XGBoost показал лучший результат")
if __name__ == "__main__":
    main()

  Эпоха 20: Loss=0.3694, Val Acc=0.8371, Val F1=0.8635
  Эпоха 40: Loss=0.3588, Val Acc=0.7841, Val F1=0.8000
  Эпоха 60: Loss=0.3329, Val Acc=0.8333, Val F1=0.8533
  Эпоха 80: Loss=0.3426, Val Acc=0.7992, Val F1=0.8179
  Эпоха 100: Loss=0.3212, Val Acc=0.8068, Val F1=0.8259
  Тестирование нейросети...
  NN: Accuracy=0.7970, F1-Score=0.8241
  Обучение CatBoost...
  CatBoost: Accuracy=0.9758, F1-Score=0.9812
  XGBoost: Accuracy=0.9697, F1-Score=0.9765
СРАВНЕНИЕ РЕЗУЛЬТАТОВ
Модель          Accuracy     F1-Score    
--------------------------------------------------
NN              0.7970       0.8241      
CatBoost        0.9758       0.9812      
XGBoost         0.9697       0.9765      

6. Сохранение моделей...
  Лучшая модель: CatBoost (F1=0.9812)
  • Деревья лучше нейросети на:
    - F1-Score: +0.1571
    - Accuracy: +0.1788
  • CatBoost показал лучший результат
